In [ ]:

{
  "cells": [
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "# EnergyCommunityEvaluationDemo\n",
        "\n",
        "This notebook creates a small synthetic energy-community dataset and then runs the real `preprocessing.py` and `run_parametric_evaluation.py` workflows in Google Colab."
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Learning goals\n",
        "\n",
        "- understand the input folder structure expected by the project,\n",
        "- run the preprocessing pipeline end to end,\n",
        "- run a small parametric evaluation with different family counts and battery sizes,\n",
        "- inspect the generated CSV and NetCDF outputs."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "import os\n",
        "import subprocess\n",
        "import sys\n",
        "\n",
        "REPO_URL = \"https://github.com/Lilol/EnergyCommunityEvaluation.git\"\n",
        "repo_dir = Path(\"/content/EnergyCommunityEvaluation\")\n",
        "if Path.cwd().name != \"EnergyCommunityEvaluation\":\n",
        "    if not repo_dir.exists():\n",
        "        subprocess.run([\"git\", \"clone\", REPO_URL, str(repo_dir)], check=True)\n",
        "    os.chdir(repo_dir)\n",
        "\n",
        "print(f\"Working directory: {Path.cwd()}\")\n",
        "subprocess.run([sys.executable, \"-m\", \"pip\", \"install\", \"-q\", \"-r\", \"requirements.txt\"], check=True)\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "from pprint import pprint\n",
        "\n",
        "from colab.demo_setup import create_demo_workspace\n",
        "\n",
        "evaluation_demo_root = Path(\"/content/EnergyCommunityEvaluationDemo\")\n",
        "evaluation_demo = create_demo_workspace(project_root=Path.cwd(), workspace_root=evaluation_demo_root)\n",
        "pprint(evaluation_demo)\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "\n",
        "config_path = Path(evaluation_demo[\"config_path\"])\n",
        "print(config_path.read_text())\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "\n",
        "try:\n",
        "    from google.colab import drive\n",
        "    drive.mount('/content/drive')\n",
        "    print('Google Drive mounted at /content/drive')\n",
        "except Exception as exc:\n",
        "    print(f'Google Drive mount skipped: {exc}')\n",
        "\n",
        "municipality = \"ColabTown\"\n",
        "local_workspace = Path(evaluation_demo[\"workspace_root\"])\n",
        "drive_workspace = Path(\"/content/drive/MyDrive/EnergyCommunityEvaluationDemo\")\n",
        "workspace_root = drive_workspace if drive_workspace.exists() else local_workspace\n",
        "print(f'Using workspace root: {workspace_root}')\n",
        "\n",
        "required_files = {\n",
        "    \"Preprocessing - required inputs\": [\n",
        "        f\"input/common/arera.csv\",\n",
        "        f\"input/common/y_ref_gse.csv\",\n",
        "        f\"input/DatabaseGSE/gse_ref_profiles.csv\",\n",
        "        f\"input/DatiComuni/{municipality}/lista_pod.csv\",\n",
        "        f\"input/DatiComuni/{municipality}/lista_impianti.csv\",\n",
        "        f\"input/DatiComuni/{municipality}/dati_bollette.csv\",\n",
        "        f\"input/DatiComuni/{municipality}/bollette_domestici.csv\",\n",
        "        f\"input/DatiComuni/{municipality}/PVGIS/*.csv or input/DatiComuni/{municipality}/PVSOL/*.csv\",\n",
        "    ],\n",
        "    \"Parametric evaluation - required preprocessing outputs\": [\n",
        "        f\"output/{municipality}/data_plants_tou.csv\",\n",
        "        f\"output/{municipality}/data_families_tou.csv\",\n",
        "        f\"output/{municipality}/data_users_tou.csv\",\n",
        "        f\"output/{municipality}/data_plants_year.csv\",\n",
        "        f\"output/{municipality}/data_families_year.csv\",\n",
        "        f\"output/{municipality}/data_users_year.csv\",\n",
        "    ],\n",
        "}\n",
        "\n",
        "def exists_or_glob(relative_item: str) -> bool:\n",
        "    if '*' in relative_item:\n",
        "        left, right = [part.strip() for part in relative_item.split(' or ')]\n",
        "        return bool(list((workspace_root / left).parent.glob((workspace_root / left).name))) or bool(list((workspace_root / right).parent.glob((workspace_root / right).name)))\n",
        "    return (workspace_root / relative_item).exists()\n",
        "\n",
        "for step_name, paths in required_files.items():\n",
        "    print(f'\\n=== {step_name} ===')\n",
        "    for rel_path in paths:\n",
        "        status = 'OK' if exists_or_glob(rel_path) else 'MISSING'\n",
        "        print(f'[{status}] {workspace_root / rel_path}')\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "\n",
        "from colab.demo_setup import run_preprocessing\n",
        "\n",
        "run_preprocessing(project_root=Path.cwd(), config_path=Path(evaluation_demo[\"config_path\"]))\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "\n",
        "from colab.demo_setup import read_output_table, validate_preprocessing_outputs\n",
        "\n",
        "workspace_root = Path(evaluation_demo[\"workspace_root\"])\n",
        "required_outputs = validate_preprocessing_outputs(workspace_root)\n",
        "print(\"Preprocessing output check passed. Required files:\")\n",
        "for name, path in required_outputs.items():\n",
        "    print(f\"- {name}: {path}\")\n",
        "\n",
        "for table_name in [\"data_users\", \"data_users_tou\", \"data_families_tou\", \"data_plants\"]:\n",
        "    print(f\"\\n=== {table_name} ===\")\n",
        "    display(read_output_table(workspace_root, table_name).head())\n",
        "\n",
        "print(\"\\n=== data_families_year (wide-format preview) ===\")\n",
        "families_year = read_output_table(workspace_root, \"data_families_year\")\n",
        "preview_columns = list(families_year.columns[:1]) + list(families_year.columns[1:13])\n",
        "display(families_year.loc[:, preview_columns].T.head(14))\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "import sys\n",
        "\n",
        "from colab.demo_setup import run_demo_evaluation\n",
        "\n",
        "config_path = Path(evaluation_demo[\"config_path\"])\n",
        "cmd = [sys.executable, \"run_parametric_evaluation.py\", \"--config\", str(config_path)]\n",
        "print(\"One-click evaluation command:\")\n",
        "print(\" \".join(cmd))\n",
        "result_files = run_demo_evaluation(project_root=Path.cwd(), config_path=config_path)\n",
        "print(f\"Generated {len(result_files)} result files.\")\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "from pathlib import Path\n",
        "\n",
        "import matplotlib.pyplot as plt\n",
        "import pandas as pd\n",
        "import xarray as xr\n",
        "\n",
        "from colab.demo_setup import find_result_files, open_output_dataarray\n",
        "\n",
        "workspace_root = Path(evaluation_demo[\"workspace_root\"])\n",
        "energy_year = open_output_dataarray(workspace_root, \"energy_year\")\n",
        "print(energy_year)\n",
        "\n",
        "result_files = result_files if \"result_files\" in globals() else find_result_files(workspace_root)\n",
        "print(\"\\nGenerated result files:\")\n",
        "for file_path in result_files:\n",
        "    print(file_path)\n",
        "\n",
        "energy_df = energy_year.to_series().dropna().reset_index()\n",
        "value_col = energy_df.columns[-1]\n",
        "top_rows = energy_df.nlargest(10, value_col)\n",
        "print(f\"\\nTop 10 scenarios by {value_col}:\")\n",
        "display(top_rows)\n",
        "\n",
        "if {\"battery_size\", \"number_of_families\"}.issubset(energy_df.columns):\n",
        "    chart_df = (\n",
        "        energy_df.groupby([\"battery_size\", \"number_of_families\"], as_index=False)[value_col]\n",
        "        .mean()\n",
        "        .sort_values(value_col, ascending=False)\n",
        "        .head(12)\n",
        "    )\n",
        "    chart_df[\"scenario\"] = chart_df.apply(\n",
        "        lambda row: f\"B={row['battery_size']}, F={row['number_of_families']}\", axis=1\n",
        "    )\n",
        "    plt.figure(figsize=(10, 4))\n",
        "    plt.bar(chart_df[\"scenario\"], chart_df[value_col])\n",
        "    plt.xticks(rotation=45, ha=\"right\")\n",
        "    plt.ylabel(value_col)\n",
        "    plt.title(\"Top scenarios (energy_year)\")\n",
        "    plt.tight_layout()\n",
        "    plt.show()\n",
        "\n",
        "if result_files:\n",
        "    sample = xr.open_dataarray(result_files[0], engine=\"netcdf4\")\n",
        "    display(sample.to_series().dropna().reset_index().head(20))\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Troubleshooting quick fixes\n",
        "\n",
        "- **Wrong working directory**: ensure the setup cell prints `/content/EnergyCommunityEvaluation` before running workflow cells.\n",
        "- **Config path mismatch**: verify `evaluation_demo[\"config_path\"]` points to `/content/EnergyCommunityEvaluationDemo/config/colab_demo_config.ini`.\n",
        "- **Missing preprocessing outputs**: rerun the preprocessing cell and then the sanity-check cell; evaluation now fails fast if key CSV files are missing.\n",
        "- **Dependency/import problems**: rerun the pip install cell (`requirements.txt`) after runtime reset.\n",
        "- **CSV delimiter confusion**: project CSVs use `;`, so use `pd.read_csv(..., sep=\";\")` when opening outputs manually.\n"
      ]
    },
    {
      "cell_type": "markdown",
      "metadata": {},
      "source": [
        "## Suggested exercises\n",
        "\n",
        "1. Change the `evaluation_parameters` entry in the generated config and compare the result files.\n",
        "2. Modify the synthetic monthly bills in `colab/demo_setup.py` to create a more winter-heavy or summer-heavy demand profile.\n",
        "3. Increase or decrease the PV size and discuss how self-consumption and emissions change.\n",
        "4. Replace the synthetic dataset with your own municipality data while keeping the same folder structure."
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "name": "python",
      "version": "3.12"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}
